# Parlay — Hugging Face only eval (no GitHub)

This notebook **only** needs a Hugging Face token (`HF_TOKEN Colab secret`) and a **GPU** runtime. After the first run of the **Install dependencies** cell, use **Runtime → Restart session** and run all cells from the top (needed so the reinstalled `torch` / `torchvision` stack is loaded). If you change packages or see `torchao`, `peft`, or `BloomPreTrainedModel` import errors, restart again.

It will:

1. Download `episodes_v2.jsonl` from the dataset [sh4shv4t/parlay-episodes](https://huggingface.co/datasets/sh4shv4t/parlay-episodes)
2. Keep rows with `split == "eval"`
3. Load **Qwen2.5-1.5B-Instruct**, [sh4shv4t/parlay-sft-1-5b](https://huggingface.co/sh4shv4t/parlay-sft-1-5b), and [sh4shv4t/parlay-grpo-1-5b](https://huggingface.co/sh4shv4t/parlay-grpo-1-5b) (LoRA adapters on top of the base)
4. For each episode, generate one JSON reply in the same chat style as training (`apply_chat_template`), parse `offer_amount`, and score **terminal deal efficiency** (GAMMA=100) with buyer- vs seller-AI ZOPA logic (same rules as the Parlay `reward_fn` for efficiency)
5. Print means and a JSON blob you can paste into `results/eval_results.json`

This is a **single-step** policy probe on the first user turn (like `training/evaluate.py` on GPU), not a full multi-turn OpenEnv roll-out.

In [ ]:
# @title 1) Install dependencies
%%capture
# Reinstall torch+torchvision+torchaudio from ONE CUDA index (fixes mismatched cu130/cu128 and the bogus BloomPreTrainedModel peft error). If cu130 wheels fail, switch URL to .../whl/cu128.
%pip install -q --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
%pip install -q -U transformers accelerate peft bitsandbytes safetensors huggingface_hub sentencepiece
%pip install -q -U "torchao>=0.16.0"

In [ ]:
# @title 2) HF_TOKEN (only secret you need)
import os
from google.colab import userdata

HF_TOKEN = (userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN") or "").strip()
if not HF_TOKEN:
    raise RuntimeError(
        "Set Colab secret HF_TOKEN: open the key icon → add HF_TOKEN with a read token from huggingface.co/settings/tokens"
    )
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print("HF_TOKEN: OK (length)", len(HF_TOKEN))


In [ ]:
# @title 3) Config — Hub IDs (defaults match the Parlay README)
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_ADAPTER = "sh4shv4t/parlay-sft-1-5b"
GRPO_ADAPTER = "sh4shv4t/parlay-grpo-1-5b"
DATASET_REPO = "sh4shv4t/parlay-episodes"
DATASET_FILE = "episodes_v2.jsonl"
N_EVAL = 50  # set smaller (e.g. 10) for a quick smoke test

import subprocess
import torch
if not torch.cuda.is_available():
    print("WARNING: no GPU — inference will be very slow; use Runtime → Change runtime type → GPU.")
else:
    subprocess.run(["nvidia-smi", "-L"], check=False)


In [ ]:
# @title 4) Load eval rows from the Hub dataset (JSONL, no local clone)
import json
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id=DATASET_REPO,
    filename=DATASET_FILE,
    repo_type="dataset",
    token=HF_TOKEN,
)
print("Downloaded:", path)

rows = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        if r.get("split") == "eval":
            rows.append(r)
if not rows:
    raise RuntimeError("No rows with split=eval in JSONL — check the dataset file name / version on the Hub.")
rows = rows[:N_EVAL]
print("Eval rows:", len(rows), "(capped to N_EVAL)")

In [ ]:
# @title 5) Scoring + prompt (matches Parlay `prompts_qwen` + efficiency reward semantics)
import re
import json as _json
from typing import Any

GAMMA = 100.0
BUYER_AI = frozenset({"hiring_package", "acquisition_term_sheet"})


def _first_user_content(conversation) -> str:
    if not isinstance(conversation, list):
        return (
            'Please make your opening offer. Reply in valid JSON: '
            '{"utterance": "...", "offer_amount": <number or null>, "tactical_move": <string or null>}'
        )
    for turn in conversation:
        if not isinstance(turn, dict):
            continue
        if turn.get("role") in ("user", "negotiator"):
            c = str(turn.get("content", "")).strip()
            if c:
                return c
    return (
        'Please make your opening offer. Reply in valid JSON: '
        '{"utterance": "...", "offer_amount": <number or null>, "tactical_move": <string or null>}'
    )


def build_generation_prompt(rec: dict, tokenizer) -> str:
    system_msg = str(rec.get("prompt", "")).strip()
    user_msg = _first_user_content(rec.get("conversation", []))
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    eot = str(
        bytes((60, 124, 105, 109, 95, 101, 110, 100, 124, 62)), "ascii"
    )
    return (
        f"<|im_start|>system\n{system_msg}{eot}\n"
        f"<|im_start|>user\n{user_msg}{eot}\n"
        f"<|im_start|>assistant\n"
    )


def parse_offer(text: str) -> float:
    t = (text or "").replace("```json", "").replace("```", "").strip()
    m = re.search(r"\{[\s\S]*\}", t)
    if not m:
        return 0.0
    try:
        d = _json.loads(m.group(0))
        v = d.get("offer_amount")
        if v is None:
            return 0.0
        return float(v)
    except Exception:
        return 0.0


def efficiency_reward(offer: float, rec: dict) -> float:
    """Parlay-style E in [0,1] * GAMMA for terminal deal efficiency."""
    batna_seller = float(rec.get("batna_seller", 0) or 0)
    batna_buyer = float(rec.get("batna_buyer", batna_seller) or batna_seller)
    zopa = max(1.0, batna_buyer - batna_seller)
    sid = str(rec.get("scenario_id", "") or "")
    is_buyer = sid in BUYER_AI
    if offer <= 0:
        return 0.0
    if is_buyer:
        e = max(0.0, min(1.0, (batna_buyer - offer) / zopa))
    else:
        e = max(0.0, min(1.0, (offer - batna_seller) / zopa))
    return e * GAMMA


In [ ]:
# @title 6) Load models (4-bit on GPU) — base, SFT, GRPO
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import hf_hub_download
import json as _json


def load_tokenizer_for(repo_id: str):
    # Prefer adapter repo tokenizer; Hub adapter repos usually ship tokenizer config
    return AutoTokenizer.from_pretrained(
        repo_id, trust_remote_code=True, token=HF_TOKEN
    )


def is_adapter_repo(repo_id: str) -> bool:
    try:
        hf_hub_download(repo_id=repo_id, filename="adapter_config.json", token=HF_TOKEN)
        return True
    except Exception:
        return False


def load_causal(hub_id: str, use_4bit: bool = True):
    use_bnb = use_4bit and torch.cuda.is_available()
    common = dict(trust_remote_code=True, token=HF_TOKEN)
    if use_bnb:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        mkw = {**common, "quantization_config": bnb, "device_map": "auto"}
    else:
        mkw = {
            **common,
            "torch_dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            "device_map": "auto" if torch.cuda.is_available() else None,
        }
    if is_adapter_repo(hub_id):
        cfg_p = Path(hf_hub_download(repo_id=hub_id, filename="adapter_config.json", token=HF_TOKEN))
        ac = _json.loads(cfg_p.read_text(encoding="utf-8"))
        base = ac.get("base_model_name_or_path", BASE_MODEL)
        base_m = AutoModelForCausalLM.from_pretrained(base, **mkw)
        m = PeftModel.from_pretrained(base_m, hub_id, token=HF_TOKEN)
    else:
        m = AutoModelForCausalLM.from_pretrained(hub_id, **mkw)
    m.eval()
    return m

print("Load helpers: OK")

In [ ]:
# @title 7) Run evaluation (reload tokenizer per checkpoint for matching templates)
import gc

SPECS = [
    ("base", BASE_MODEL),
    ("sft", SFT_ADAPTER),
    ("grpo", GRPO_ADAPTER),
]


@torch.inference_mode()
def run_eval_for_repo(name: str, repo_id: str) -> float:
    tok = load_tokenizer_for(repo_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = None
    try:
        model = load_causal(repo_id, use_4bit=True)
        rews = []
        for i, rec in enumerate(rows):
            prompt = build_generation_prompt(rec, tok)
            dev = next(model.parameters()).device
            batch = tok(
                prompt, return_tensors="pt", max_length=4096, truncation=True
            )
            batch = {k: v.to(dev) for k, v in batch.items()}
            out = model.generate(
                **batch,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tok.pad_token_id,
            )
            gen = out[0][batch["input_ids"].shape[-1] :]
            text = tok.decode(gen, skip_special_tokens=True)
            offer = parse_offer(text)
            rews.append(efficiency_reward(offer, rec))
        return sum(rews) / max(len(rews), 1)
    finally:
        if model is not None:
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if torch.cuda.is_available():
            torch.cuda.synchronize()


results = {}
for name, rid in SPECS:
    print("Evaluating:", name, rid)
    m = run_eval_for_repo(name, rid)
    results[name] = m
    print(f"  mean reward (eff proxy): {m:.3f}")

print("\n--- Summary ---", results)


In [ ]:
# @title 8) Build `eval_results.json` (copy into your repo: results/eval_results.json)
import json
out = {
    "base_mean_reward": results["base"],
    "sft_mean_reward": results.get("sft"),
    "grpo_mean_reward": results.get("grpo"),
    "n_eval": len(rows),
    "dataset": DATASET_REPO,
    "data_file": DATASET_FILE,
    "models": {
        "base": BASE_MODEL,
        "sft": SFT_ADAPTER,
        "grpo": GRPO_ADAPTER,
    },
}
print(json.dumps(out, indent=2))
from google.colab import files
path = "/content/eval_results.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2)
files.download(path)
print("Downloaded eval_results.json — add random_mean_reward manually (e.g. from random_baseline) if needed.")